# Mini-projeto 1 - Fase 2: CNN para classificação do CIFAR-10

Continuidade da Fase 1 (MLP, ver `../fase1-mlp/`). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `cnn_cifar10` em `../src/`, seguindo exatamente o mesmo padrão da Fase 1 — o notebook fica focado em **definir experimentos e reportar resultados**, não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de uma CNN no CIFAR-10 com hiperparâmetros configuráveis (nº/tamanho de filtros, kernel size, stride, padding, pooling, dropout, taxa de aprendizagem, além dos já cobertos na Fase 1: ativação, otimizador, função de erro).
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação direta com o melhor resultado do MLP (Fase 1: ensemble `final` = 0.6135 de acurácia) — ver `../../fase1-mlp/README.md`.
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/cnn_cifar10/checkpointing.py`.

**Recomendação forte: rode este notebook no Google Colab com GPU** (Ambiente de execução > Alterar tipo de ambiente de execução > GPU). CNN é bem mais lenta que MLP em CPU, e o ganho de GPU aqui é de 10-50x+.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase2-cnn/`).
- **Google Colab**: a célula abaixo clona o repositório e instala o pacote automaticamente. O repositório é **privado**, então precisa de um GitHub Personal Access Token (PAT) — ver instruções abaixo, só precisa configurar uma vez.

### Gerar o token (uma vez só)

1. No GitHub: `Settings > Developer settings > Personal access tokens > Fine-grained tokens > Generate new token`.
2. Repository access: `Only select repositories` > `redes-neurais`.
3. Permissions: `Contents` = `Read-only` (só precisa ler/clonar, não escrever).
4. Defina uma expiração (ex.: 90 dias) e gere o token — copie o valor (`github_pat_...`), ele só aparece uma vez.

### Guardar no Colab (uma vez por navegador/conta)

1. No Colab, clique no ícone de chave (🔑 **Secrets**) na barra lateral esquerda.
2. `Add new secret` → nome `GITHUB_TOKEN`, valor = o token copiado acima.
3. Ative o toggle "Notebook access" para este notebook.

A célula abaixo lê o secret automaticamente; se não encontrar, pede o token via prompt (não fica salvo em lugar nenhum do notebook).

In [10]:
#@title Setup (Colab ou local)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import shutil

    BRANCH = "feat/cnn"  #@param {type:"string"}
    # Ajuste para "main" (ou o branch que estiver usando) quando o trabalho da
    # Fase 2 for mesclado - não precisa editar mais nada além desta linha.
    REPO_PATH = "github.com/jpbezerra/redes-neurais.git"
    REPO_DIR = Path("/content/redes-neurais")

    # Se uma tentativa anterior de clone falhou no meio (ex.: token errado),
    # a pasta pode existir mas sem ser um repositorio git valido - nesse caso
    # apagamos e clonamos de novo em vez de só tentar "git pull" nela.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if not REPO_DIR.exists():
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            import getpass
            token = getpass.getpass("Repositorio privado - cole seu GitHub Personal Access Token: ")
        clone_url = f"https://{token}@{REPO_PATH}"
        # o token fica salvo em .git/config só dentro desta VM efêmera do Colab
        # (destruída ao fim da sessão) - necessário para o "git pull" funcionar
        # de novo mais tarde na mesma sessão, sem pedir o token de novo.
        !git clone -q -b {BRANCH} {clone_url} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q origin {BRANCH}

    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase2-cnn"
    assert (PROJECT_ROOT / "pyproject.toml").exists(), (
        f"Clone parece ter falhado (pyproject.toml nao encontrado em {PROJECT_ROOT}). "
        f"Confira: (1) BRANCH='{BRANCH}' e o branch certo, (2) o token em Secrets "
        "(GITHUB_TOKEN) esta valido - depois 'Ambiente de execucao > Reiniciar sessao' "
        "e rode esta celula de novo."
    )
    %pip install -q -e {PROJECT_ROOT}
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase2-cnn/

sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("PROJECT_ROOT:", PROJECT_ROOT)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cnn-cifar10 (pyproject.toml) ... done
PROJECT_ROOT: /content/redes-neurais/miniprojeto/fase2-cnn


In [11]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from cnn_cifar10.config import ExperimentConfig
from cnn_cifar10.data import get_dataloaders, CLASSES
from cnn_cifar10.train import fit, fit_or_load
from cnn_cifar10.checkpointing import load_all_metadata
from cnn_cifar10.utils import set_seed, get_device

In [12]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)
if device.type == "cpu":
    print("Aviso: sem GPU disponível. CNN em CPU é bem mais lenta — considere rodar no Colab.")

# No Colab/Linux, num_workers > 0 acelera o carregamento com augmentation
# (no Windows local, exige if __name__ == '__main__': em scripts, então mantemos 0 lá).
NUM_WORKERS = 2 if IN_COLAB else 0

Usando dispositivo: cuda


## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência do professor (`temp/CIFAR10_with_CNNs.ipynb`, adaptação do LeNet-5): 2 blocos convolucionais (32 e 64 filtros, kernel 3x3, padding 1, stride 1) + max pooling 2x2 após cada bloco, seguidos de cabeça densa `120 -> 84 -> 10`. ReLU, Adam, entropia cruzada, sem regularização — ponto de partida para a busca guiada, do mesmo jeito que a Fase 1 partiu do baseline MLP `[64,128,64]`.

In [13]:
baseline_config = ExperimentConfig(
    run_name="baseline",
    conv_channels=(32, 64),
    kernel_size=3,
    stride=1,
    padding=1,
    pool_size=2,
    fc_layers=(120, 84),
    activation="relu",
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    batch_size=32,
    num_epochs=40,
    patience=5,
    notes="Baseline equivalente ao notebook de referencia (2 conv + 2 pool + 3 fc), ponto de partida da busca guiada da Fase 2.",
    tags=["baseline"],
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
    num_workers=NUM_WORKERS,
    augment=baseline_config.augment,
    normalization=baseline_config.normalization,
)

result_baseline = fit_or_load(
    baseline_config, train_loader, val_loader, test_loader, device,
    class_names=list(CLASSES), results_dir=RESULTS_DIR,
)
result_baseline["test_scores"]

100%|██████████| 170M/170M [24:04<00:00, 118kB/s]


treinando 'baseline':   0%|          | 0/40 [00:00<?, ?it/s]

Época 1/40 | train_loss=1.3641 | val_loss=1.0566 | val_acc=0.6272
Época 2/40 | train_loss=0.9682 | val_loss=0.9147 | val_acc=0.6770
Época 3/40 | train_loss=0.7919 | val_loss=0.8745 | val_acc=0.6900
Época 4/40 | train_loss=0.6640 | val_loss=0.8211 | val_acc=0.7260
Época 5/40 | train_loss=0.5528 | val_loss=0.8311 | val_acc=0.7314
Época 6/40 | train_loss=0.4561 | val_loss=0.9022 | val_acc=0.7222
Época 7/40 | train_loss=0.3616 | val_loss=0.9661 | val_acc=0.7104
Época 8/40 | train_loss=0.2887 | val_loss=1.0826 | val_acc=0.7142
Época 9/40 | train_loss=0.2280 | val_loss=1.2168 | val_acc=0.7078
Early stopping acionado.
Métricas no conjunto de teste: {'accuracy': 0.7148, 'balanced_accuracy': np.float64(0.7148), 'precision': 0.7129270673384086, 'recall': 0.7148, 'f1_score': 0.7122810504679414}
Acurácia por classe: {'airplane': 0.758, 'automobile': 0.863, 'bird': 0.588, 'cat': 0.495, 'deer': 0.652, 'dog': 0.608, 'frog': 0.85, 'horse': 0.758, 'ship': 0.832, 'truck': 0.744}
[model-saver] Modelo sal

{'accuracy': 0.7148,
 'balanced_accuracy': np.float64(0.7148),
 'precision': 0.7129270673384086,
 'recall': 0.7148,
 'f1_score': 0.7122810504679414}

## 2. Leva 1 — variações isoladas (uma alavanca por vez)

Mesmo espírito da leva 1 do MLP: cada configuração muda **um** hiperparâmetro
em relação ao baseline, para medir o efeito isolado antes de combinar
vencedores em rodadas sucessivas (busca gulosa, como nas levas 2-8 do MLP).
Cobre todos os parâmetros pedidos no enunciado da Fase 2: tamanho da rede,
kernel size, stride, padding, dropout, pooling e taxa de aprendizagem — mais
batch norm e augmentation como bônus (mesma cobertura extra feita no MLP).

`num_epochs=30`/`patience=5` (menor que o baseline) para essa leva exploratória
rodar mais rápido — se algum candidato for promissor, pode retreinar depois
com mais épocas.

In [14]:
candidate_configs = [
    ExperimentConfig(
        run_name="kernel_5",
        conv_channels=(32, 64), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Kernel 5x5 em vez de 3x3 (padding=2 mantem o tamanho espacial 'same').",
    ),
    ExperimentConfig(
        run_name="stride_2_no_pool",
        conv_channels=(32, 64), kernel_size=3, stride=2, padding=1, pool_size=1,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Substitui o pooling por stride=2 na propria convolucao (pool_size=1 = sem pooling extra).",
    ),
    ExperimentConfig(
        run_name="padding_0",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=0, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Convolucao 'valid' (sem padding) em vez de 'same' (padding=1) do baseline.",
    ),
    ExperimentConfig(
        run_name="pool_4",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=4,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Janela de pooling maior (4x4 em vez de 2x2) apos cada bloco convolucional.",
    ),
    ExperimentConfig(
        run_name="deeper_3conv",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Rede maior: 3 blocos convolucionais (32/64/128 filtros) em vez de 2.",
    ),
    ExperimentConfig(
        run_name="dropout_03",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), dropout=0.3, num_epochs=30, patience=5,
        notes="Dropout 0.3 (Dropout2d nos blocos conv + Dropout na cabeca densa) sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="batch_norm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), batch_norm=True, num_epochs=30, patience=5,
        notes="Adiciona BatchNorm2d apos cada convolucao, sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="lr_low",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-4, num_epochs=30, patience=5,
        notes="Learning rate 2x menor que o baseline (1e-3 -> 5e-4).",
    ),
    ExperimentConfig(
        run_name="lr_high",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-3, num_epochs=30, patience=5,
        notes="Learning rate 5x maior que o baseline (1e-3 -> 5e-3).",
    ),
    ExperimentConfig(
        run_name="augment",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=30, patience=5,
        notes="Data augmentation (crop+flip) no treino, mesma transform da Fase 1.",
    ),
    # Adicione outras variacoes conforme os experimentos forem sendo decididos.
]

In [15]:
experiment_results = {baseline_config.run_name: result_baseline}
for config in candidate_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

treinando 'kernel_5':   0%|          | 0/30 [00:00<?, ?it/s]

Época 1/30 | train_loss=1.3761 | val_loss=1.0773 | val_acc=0.6210
Época 2/30 | train_loss=0.9798 | val_loss=0.9474 | val_acc=0.6682
Época 3/30 | train_loss=0.8017 | val_loss=0.8587 | val_acc=0.6958
Época 4/30 | train_loss=0.6711 | val_loss=0.8480 | val_acc=0.7094
Época 5/30 | train_loss=0.5650 | val_loss=0.8566 | val_acc=0.7088
Época 6/30 | train_loss=0.4696 | val_loss=0.9816 | val_acc=0.7060
Época 7/30 | train_loss=0.3946 | val_loss=0.9509 | val_acc=0.7046
Época 8/30 | train_loss=0.3201 | val_loss=1.0678 | val_acc=0.7014
Época 9/30 | train_loss=0.2715 | val_loss=1.1455 | val_acc=0.6986
Early stopping acionado.
Métricas no conjunto de teste: {'accuracy': 0.7035, 'balanced_accuracy': np.float64(0.7034999999999999), 'precision': 0.7066374910321646, 'recall': 0.7035, 'f1_score': 0.7006502013981192}
Acurácia por classe: {'airplane': 0.81, 'automobile': 0.822, 'bird': 0.605, 'cat': 0.443, 'deer': 0.741, 'dog': 0.532, 'frog': 0.776, 'horse': 0.8, 'ship': 0.781, 'truck': 0.725}
[model-saver] 

treinando 'stride_2_no_pool':   0%|          | 0/30 [00:00<?, ?it/s]

Época 1/30 | train_loss=1.4657 | val_loss=1.2343 | val_acc=0.5656
Época 2/30 | train_loss=1.1122 | val_loss=1.0714 | val_acc=0.6166
Época 3/30 | train_loss=0.9286 | val_loss=0.9947 | val_acc=0.6438
Época 4/30 | train_loss=0.7769 | val_loss=1.0364 | val_acc=0.6462
Época 5/30 | train_loss=0.6485 | val_loss=1.0175 | val_acc=0.6592
Época 6/30 | train_loss=0.5277 | val_loss=1.1268 | val_acc=0.6528
Época 7/30 | train_loss=0.4203 | val_loss=1.2312 | val_acc=0.6442
Época 8/30 | train_loss=0.3358 | val_loss=1.3373 | val_acc=0.6516
Early stopping acionado.
Métricas no conjunto de teste: {'accuracy': 0.643, 'balanced_accuracy': np.float64(0.643), 'precision': 0.656135258511737, 'recall': 0.643, 'f1_score': 0.6442358355161343}
Acurácia por classe: {'airplane': 0.715, 'automobile': 0.735, 'bird': 0.443, 'cat': 0.628, 'deer': 0.591, 'dog': 0.429, 'frog': 0.697, 'horse': 0.708, 'ship': 0.752, 'truck': 0.732}
[model-saver] Modelo salvo em /content/redes-neurais/miniprojeto/fase2-cnn/results/cnn_stride

treinando 'padding_0':   0%|          | 0/30 [00:00<?, ?it/s]

...

Época 1/30 | train_loss=1.4649 | val_loss=1.1954 | val_acc=0.5778
Época 2/30 | train_loss=1.0913 | val_loss=1.0076 | val_acc=0.6392
Época 3/30 | train_loss=0.9197 | val_loss=0.9165 | val_acc=0.6708
Época 4/30 | train_loss=0.8044 | val_loss=0.8526 | val_acc=0.6976
Época 5/30 | train_loss=0.7107 | val_loss=0.8157 | val_acc=0.7176
Época 6/30 | train_loss=0.6250 | val_loss=0.8138 | val_acc=0.7188
Época 7/30 | train_loss=0.5416 | val_loss=0.8260 | val_acc=0.7208
Época 8/30 | train_loss=0.4637 | val_loss=0.8673 | val_acc=0.7212
Época 9/30 | train_loss=0.3953 | val_loss=0.9080 | val_acc=0.7208
Época 10/30 | train_loss=0.3280 | val_loss=0.9375 | val_acc=0.7238
Época 11/30 | train_loss=0.2729 | val_loss=1.0465 | val_acc=0.7192
Early stopping acionado.
Métricas no conjunto de teste: {'accuracy': 0.7141, 'balanced_accuracy': np.float64(0.7141000000000001), 'precision': 0.718703520840902, 'recall': 0.7141, 'f1_score': 0.7128678189521056}
Acurácia por classe: {'airplane': 0.666, 'automobile': 0.846

treinando 'lr_high':   0%|          | 0/30 [00:00<?, ?it/s]

Época 1/30 | train_loss=1.5655 | val_loss=1.3643 | val_acc=0.5038
Época 2/30 | train_loss=1.3157 | val_loss=1.2692 | val_acc=0.5474
Época 3/30 | train_loss=1.2212 | val_loss=1.2018 | val_acc=0.5658
Época 4/30 | train_loss=1.1634 | val_loss=1.1915 | val_acc=0.5878
Época 5/30 | train_loss=1.1217 | val_loss=1.1348 | val_acc=0.6062
Época 6/30 | train_loss=1.0888 | val_loss=1.1609 | val_acc=0.5958
Época 7/30 | train_loss=1.0628 | val_loss=1.1404 | val_acc=0.5940
Época 8/30 | train_loss=1.0290 | val_loss=1.1341 | val_acc=0.6070
Época 9/30 | train_loss=1.0146 | val_loss=1.1946 | val_acc=0.5852
Época 10/30 | train_loss=0.9843 | val_loss=1.1448 | val_acc=0.6128
Época 11/30 | train_loss=0.9688 | val_loss=1.1463 | val_acc=0.6054
Época 12/30 | train_loss=0.9513 | val_loss=1.1189 | val_acc=0.6252
Época 13/30 | train_loss=0.9363 | val_loss=1.1201 | val_acc=0.6178
Época 14/30 | train_loss=0.9119 | val_loss=1.0990 | val_acc=0.6158
Época 15/30 | train_loss=0.8945 | val_loss=1.1276 | val_acc=0.6148
Époc

treinando 'augment':   0%|          | 0/30 [00:00<?, ?it/s]

Época 1/30 | train_loss=1.5902 | val_loss=1.2750 | val_acc=0.5310
Época 2/30 | train_loss=1.2676 | val_loss=1.0652 | val_acc=0.6174
Época 3/30 | train_loss=1.1192 | val_loss=1.0431 | val_acc=0.6392
Época 4/30 | train_loss=1.0253 | val_loss=0.8935 | val_acc=0.6856
Época 5/30 | train_loss=0.9672 | val_loss=0.8684 | val_acc=0.6948
Época 6/30 | train_loss=0.9234 | val_loss=0.8589 | val_acc=0.6986
Época 7/30 | train_loss=0.8845 | val_loss=0.8175 | val_acc=0.7142
Época 8/30 | train_loss=0.8575 | val_loss=0.8280 | val_acc=0.7112
Época 9/30 | train_loss=0.8329 | val_loss=0.8012 | val_acc=0.7244
Época 10/30 | train_loss=0.8097 | val_loss=0.7837 | val_acc=0.7258
Época 11/30 | train_loss=0.7967 | val_loss=0.7580 | val_acc=0.7328
Época 12/30 | train_loss=0.7832 | val_loss=0.7473 | val_acc=0.7448
Época 13/30 | train_loss=0.7689 | val_loss=0.7643 | val_acc=0.7368
Época 14/30 | train_loss=0.7517 | val_loss=0.7416 | val_acc=0.7484
Época 15/30 | train_loss=0.7459 | val_loss=0.7174 | val_acc=0.7546
Époc

In [16]:
#@title Tabela comparativa dos experimentos
comparison = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison

,accuracy,balanced_accuracy,precision,recall,f1_score
augment,0.7635,0.7635,0.771671,0.7635,0.764562
deeper_3conv,0.7394,0.7394,0.741505,0.7394,0.738148
batch_norm,0.7336,0.7336,0.734223,0.7336,0.732073
baseline,0.7148,0.7148,0.712927,0.7148,0.712281
lr_low,0.7141,0.7141,0.718704,0.7141,0.712868
dropout_03,0.7130,0.7130,0.709245,0.7130,0.709568
padding_0,0.7119,0.7119,0.716136,0.7119,0.711563
pool_4,0.7117,0.7117,0.722152,0.7117,0.714032
kernel_5,0.7035,0.7035,0.706637,0.7035,0.700650
stride_2_no_pool,0.6430,0.6430,0.656135,0.6430,0.644236


## 3. Próximas rodadas

Depois de ver os resultados da leva 1 acima, combine os hiperparâmetros
vencedores em rodadas sucessivas (busca gulosa), como nas levas 2-8 do MLP —
cada rodada informada pelo resultado da anterior. Se as rodadas ficarem
grandes/lentas demais para caber numa célula, considere mover para um
`scripts/run_experiments.py` (ver `../fase1-mlp/scripts/run_experiments.py`
como referência de estrutura).

In [17]:
#@title Comparar todas as execuções salvas
df = load_all_metadata(RESULTS_DIR)
if not df.empty:
    cols = [c for c in ["run_name", "metrics.test_accuracy", "metrics.test_f1_score", "metrics.epochs_trained"] if c in df.columns]
    display(df[cols].sort_values("metrics.test_accuracy", ascending=False) if cols else df)
else:
    print("Nenhuma execucao salva ainda em", RESULTS_DIR)

,run_name,metrics.test_accuracy,metrics.test_f1_score,metrics.epochs_trained
0,augment,0.7635,0.764562,30
3,deeper_3conv,0.7394,0.738148,10
2,batch_norm,0.7336,0.732073,12
1,baseline,0.7148,0.712281,9
7,lr_low,0.7141,0.712868,11
4,dropout_03,0.7130,0.709568,30
8,padding_0,0.7119,0.711563,11
9,pool_4,0.7117,0.714032,15
5,kernel_5,0.7035,0.700650,9
10,stride_2_no_pool,0.6430,0.644236,8


## 4. Baixar `results/` para trazer de volta ao repositório local

**Só necessário no Colab** (local já grava direto em `../results/`). Os
resultados salvos em `/content/redes-neurais/.../results/` somem quando a
sessão do Colab reinicia — rode esta célula ao final de cada sessão de
experimentos para não perder o trabalho. Ela gera um `.zip` e baixa para a
pasta de Downloads do seu computador; depois é só extrair por cima da pasta
`results/` local (ou pedir para o Claude fazer isso) e dar commit/push.

In [18]:
#@title Baixar results/ (Colab)
if IN_COLAB:
    import shutil
    from google.colab import files

    zip_path = shutil.make_archive("/content/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path)
    files.download(zip_path)
else:
    print("Local: results/ ja esta em", RESULTS_DIR, "- nao precisa baixar nada.")

Gerado: /content/results_cnn.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>